# ISM 6564 — Assignment 4: does reading in order pay for itself?

**Northwind Outfitters, review triage.** Every product review is scored by a
sentiment classifier: negative reviews open a support ticket, positive ones are
sampled for marketing quotes. The classifier is last month's kind of model — a
bag of words — and the support team has found reviews it gets backwards. Your
job is to build the Week 4 alternative, measure where it helps and what it
costs, and recommend what Northwind should run.

The full brief, the rubric, and the submission list are on the **assignment
page in Canvas**. This notebook needs nothing else: the data loads itself from
the public course data repository.

How this notebook works:

- Cells marked **`GIVEN`** are provided. Run them; do not modify them.
- Cells marked **`TODO`** are yours. Each starts with `None` — replace it with
  your work.
- The cell after each TODO **checks** your work and prints a clear message
  while anything is missing, so the notebook always runs top to bottom. Do not
  delete the check cells.
- Italic text in a *Your answer* cell is a placeholder: replace it with your
  own writing. Written answers must be your own words (course AI policy).

---
## Setup and data

In [35]:
# GIVEN — imports, seeds, and where the data lives.
import random
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

set_seed()
DATA = "https://raw.githubusercontent.com/prof-tcsmith/6564F26-DATA/main/week-04-sequence-models/data/"
SLICES = ["simple", "contrast", "scope_short", "scope_long"]
print(f"torch {torch.__version__} — everything here runs on an ordinary laptop CPU")

torch 2.13.0+cpu — everything here runs on an ordinary laptop CPU


In [36]:
# GIVEN — load the reviews. The split is fixed in the file: do not re-split it.
reviews = pd.read_csv(DATA + "reviews.csv")
train_df = reviews[reviews["split"] == "train"].reset_index(drop=True)
val_df = reviews[reviews["split"] == "val"].reset_index(drop=True)
test_df = reviews[reviews["split"] == "test"].reset_index(drop=True)

print(f"{len(reviews)} reviews: train {len(train_df)}, validation {len(val_df)}, test {len(test_df)}\n")
print("test reviews per slice:", test_df["slice"].value_counts()[SLICES].to_dict(), "\n")
for s in SLICES:
    row = test_df[test_df["slice"] == s].iloc[0]
    print(f"[{s:<11}] {'positive' if row['label'] == 1 else 'negative'}: {row['text']}")

4837 reviews: train 3164, validation 469, test 1204

test reviews per slice: {'simple': 273, 'contrast': 296, 'scope_short': 336, 'scope_long': 299} 

[simple     ] positive: the companion app is responsive and clean
[contrast   ] negative: ordered this in june the mounting bracket lines up perfectly but the keys started sticking almost immediately model number lc four seven five
[scope_short] positive: buying for a home office it is not true that the hinge went loose within a month updating this review later
[scope_long ] negative: bought during the autumn sale it is not true that even allowing for the most recent firmware update compared with the previous model i owned the sound is surprisingly rich ordered through the app


Each review's `slice` says which kind of word order decides its label:

| `slice` | What it looks like |
|---|---|
| `simple` | one opinion, no tricks |
| `contrast` | *"A **but** B"* — the label follows **B**; every review also appears in the opposite order, with the opposite label |
| `scope_short` | a frame that may contain **not** (*"it is not true that…"*), then the opinion a few words later |
| `scope_long` | the same, with 15–20 filler words between the *not* and the opinion |

In [37]:
# GIVEN — scoring helpers. Every model's test results go into one table.
results = []

def slice_report(name, predictions, train_seconds, n_params):
    correct = predictions == test_df["label"].values
    row = {"model": name, "overall": correct.mean()}
    for s in SLICES:
        row[s] = correct[test_df["slice"].values == s].mean()
    row["train seconds"] = round(train_seconds, 2)
    row["learned numbers"] = n_params
    results.append(row)

def show_results():
    print(pd.DataFrame(results).set_index("model").round(4).to_string())

---
## Task 1 — The bag of words, and exactly where it fails

Two versions of the model Northwind runs today: TF-IDF over **single words**,
and TF-IDF over **single words plus word pairs** (bigrams: every two adjacent
words also count as a feature, so *not good* gets a column of its own). Both
are given — run them and read the table.

In [38]:
# GIVEN — the two bag-of-words models.
def fit_bag_of_words(ngram_range):
    t0 = time.perf_counter()
    vec = TfidfVectorizer(ngram_range=ngram_range)
    model = LogisticRegression(max_iter=2000).fit(vec.fit_transform(train_df["text"]), train_df["label"])
    seconds = time.perf_counter() - t0
    predictions = model.predict(vec.transform(test_df["text"]))
    n_params = model.coef_.size + model.intercept_.size
    return vec, model, predictions, seconds, n_params

words_vec, words_model, words_pred, words_secs, words_params = fit_bag_of_words((1, 1))
pairs_vec, pairs_model, pairs_pred, pairs_secs, pairs_params = fit_bag_of_words((1, 2))
slice_report("bag of words: single words", words_pred, words_secs, words_params)
slice_report("bag of words: + word pairs", pairs_pred, pairs_secs, pairs_params)
show_results()

                            overall  simple  contrast  scope_short  scope_long  train seconds  learned numbers
model                                                                                                         
bag of words: single words   0.6453     1.0    0.5000       0.5327      0.5920           0.06              289
bag of words: + word pairs   0.7533     1.0    0.9291       0.5357      0.5987           0.09             1350


In [39]:
# GIVEN — look closely at one contrast pair from the test set.
contrast = test_df[test_df["slice"] == "contrast"]
key = contrast["text"].map(lambda t: " ".join(sorted(t.split())))
pair = contrast[key == key.iloc[0]]
for _, row in pair.iterrows():
    p = words_model.predict_proba(words_vec.transform([row["text"]]))[0, 1]
    print(f"true: {'positive' if row['label'] == 1 else 'negative'} | single-words model P(positive) = {p:.4f}")
    print(f"   {row['text']}\n")

true: negative | single-words model P(positive) = 0.5505
   ordered this in june the mounting bracket lines up perfectly but the keys started sticking almost immediately model number lc four seven five

true: positive | single-words model P(positive) = 0.5505
   ordered this in june the keys started sticking almost immediately but the mounting bracket lines up perfectly model number lc four seven five



### Your answer — Task 1

*A. The single-words model's accuracy on contrast is 0.5, the number is exactly this because the words are counted, not the context. The model counts the total apperance of negative versus positive vectors. This is shown in the example above where both queries are scored the same despite having opposite meanings. The first sentence complains about the 'key being immediatly stuck' while the second does not; due to the word order not accounted for. 
B. Adding bigrams help with contrast because negative terms such as 'not good' can be bunched together, instead of being seperate. However, this does not help the overall scope because in the end, bigram only deals with two words and does not account for order of sentences or larger sequence of words. *

---
## Task 2 — Build the model that reads in order

Build the Week 4 model: an **embedding** table (word ID → 64 numbers; the rows
start random and are learned from this task), an **LSTM** that reads the review
one word at a time and carries a 64-number memory, and the **head** — a linear
layer that turns the memory after the last word into two scores (negative,
positive). Two sizes, both 64 here: how many numbers describe a word, and how
many the memory carries. The words-to-IDs encoding and the training loop are
given — they are the same as in the lecture notebook's §5.

In [40]:
# GIVEN — words -> IDs, exactly as in the lecture notebook (vocabulary from TRAIN only).
tokenize = CountVectorizer().build_analyzer()
MAX_LEN = 50                                    # the longest review has 49 words

word_counts = Counter(w for text in train_df["text"] for w in tokenize(text))
vocab = {"<pad>": 0, "<unk>": 1}
for w, n in word_counts.items():
    if n >= 2:
        vocab[w] = len(vocab)

def encode(texts):
    rows = np.zeros((len(texts), MAX_LEN), dtype=np.int64)          # all <pad> to start
    for i, text in enumerate(texts):
        ids = [vocab.get(w, 1) for w in tokenize(text)][:MAX_LEN]
        rows[i, MAX_LEN - len(ids):] = ids                            # padding in FRONT
    return torch.from_numpy(rows)

X_train, X_val, X_test = encode(train_df["text"]), encode(val_df["text"]), encode(test_df["text"])
y_train = torch.tensor(train_df["label"].values)
y_val = torch.tensor(val_df["label"].values)
print(f"vocabulary: {len(vocab)} words   X_train {tuple(X_train.shape)}")

vocabulary: 290 words   X_train (3164, 50)


In [41]:
# TODO — replace None with your model class.
# It must take (vocab_size, cell=nn.LSTM) and:
#   1. turn word IDs into 64-number vectors      -> nn.Embedding(vocab_size, 64, padding_idx=0)
#   2. read them in order with a 64-number memory -> cell(64, 64, batch_first=True)
#   3. the head: turn the memory after the LAST word into 2 scores -> nn.Linear(64, 2)
# The lecture notebook's TicketReader (§5) is your template: same three parts, a smaller head.
#ReviewReader= None
class ReviewReader(nn.Module):
    def __init__(self,vocab_size,cell=nn.LSTM):
        super().__init__()
        self.embed=nn.Embedding(vocab_size, 64,padding_idx=0)#   1. turn word IDs into 64-number vectors      -> nn.Embedding(vocab_size, 64, padding_idx=0)
        self.reader = cell(64,64,batch_first=True)#   2. read them in order with a 64-number memory -> cell(64, 64, batch_first=True)
        self.head= nn.Linear(64,2)#   3. the head: turn the memory after the LAST word into 2 scores -> nn.Linear(64, 2)

    def forward(self,ids):
        word_vectors=self.embed(ids)
        outputs,final=self.reader(word_vectors)
        if isinstance(final, tuple):
            final= final[0]
        return self.head(final[-1])

# Used claude to identify why code was not working, due to _init_ having one underscore instead of two, and arguments for (self,vocab_size,cell) was in wrong order. 

In [42]:
# CHECK — Task 2 model
if ReviewReader is None:
    print("[not yet] Task 2: replace `ReviewReader = None` with your model class.")
else:
    set_seed()
    _m = ReviewReader(len(vocab))
    _out = _m(X_train[:4])
    assert tuple(_out.shape) == (4, 2), f"expected 2 scores per review, got shape {tuple(_out.shape)}"
    _emb = [m for m in _m.modules() if isinstance(m, nn.Embedding)]
    assert _emb and _emb[0].padding_idx == 0, "use nn.Embedding(..., padding_idx=0)"
    assert any(isinstance(m, nn.LSTM) for m in _m.modules()), "the default cell should be nn.LSTM"
    print(f"[ok] Task 2: your model has {sum(p.numel() for p in _m.parameters()):,} learned numbers")

[ok] Task 2: your model has 51,970 learned numbers


In [43]:
# GIVEN — the training loop from Week 3 and the lecture: five lines, validation, early stopping.
# Each batch: predict (2 scores per review) -> measure (cross-entropy against the true label)
#          -> nudge every learned number, the embedding rows included.
def train_reader(model, max_epochs=40, patience=6):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    batches = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
    best_val, best_weights, waited = 0.0, None, 0
    t0 = time.perf_counter()
    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in batches:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_acc = (model(X_val).argmax(1) == y_val).float().mean().item()
        if val_acc > best_val:
            best_val, waited = val_acc, 0
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience:
                break
    model.load_state_dict(best_weights)
    model.eval()
    return model, time.perf_counter() - t0

def predict(model, X):
    with torch.no_grad():
        return model(X).argmax(1).numpy()

def count(model):
    return sum(p.numel() for p in model.parameters())

In [44]:
# GIVEN — train your LSTM and score it on the test set.
if ReviewReader is None:
    print("[not yet] Task 2: build ReviewReader above, then re-run this cell.")
else:
    set_seed()
    lstm_model, lstm_secs = train_reader(ReviewReader(len(vocab)))
    slice_report("LSTM reader", predict(lstm_model, X_test), lstm_secs, count(lstm_model))
    show_results()

                            overall  simple  contrast  scope_short  scope_long  train seconds  learned numbers
model                                                                                                         
bag of words: single words   0.6453     1.0    0.5000       0.5327      0.5920           0.06              289
bag of words: + word pairs   0.7533     1.0    0.9291       0.5357      0.5987           0.09             1350
LSTM reader                  0.9992     1.0    1.0000       0.9970      1.0000           6.92            51970


---
## Task 3 — Where did reading in order help?

The table above now has three rows. Read it slice by slice.

### Your answer — Task 3

*-LSTM beats bag-of-words models on scope while coming close to word pairs on contrast. This is because LSTM's structure of three gates and having a context throughout the process. It beat other models in contrast by 0.08 points by scoring 100%. For scope_short,the model performed 0.46 better than the next best. For scope_long, the model performed 0.41 better than the bag-of word models. 

-They are tied in 'simple' slice. That is expected because for simple tasks the context remain the same throughout the sentence which calls for no need for long context. *

---
## Task 4 — Test it yourself

Write your own test reviews. Each **pair** must use exactly the **same words
in a different order**, so that the meaning flips — like the `contrast`
reviews above. Then see what each model makes of them.

`unknown_words(text)` lists any word the LSTM never saw in training (it would
become `<unk>`); rewrite those, using words from the example reviews.

In [45]:
# GIVEN — helpers for Task 4.
def unknown_words(text):
    return [w for w in tokenize(text) if w not in vocab]

def compare(texts):
    rows = []
    for t in texts:
        with torch.no_grad():
            lstm_p = torch.softmax(lstm_model(encode([t])), 1)[0, 1].item()
        rows.append({
            "review": t[:70] + ("…" if len(t) > 70 else ""),
            "single words": round(float(words_model.predict_proba(words_vec.transform([t]))[0, 1]), 3),
            "+ word pairs": round(float(pairs_model.predict_proba(pairs_vec.transform([t]))[0, 1]), 3),
            "LSTM": round(lstm_p, 3),
        })
    print("probability of POSITIVE from each model")
    print(pd.DataFrame(rows).to_string(index=False))

In [46]:
# TODO — write at least 3 pairs: (review, the same words re-ordered so the meaning flips).
# Example of the shape (write your own, do not reuse this one):
#   ("the sound is surprisingly rich but the hinge went loose within a month",
#    "the hinge went loose within a month but the sound is surprisingly rich")
my_pairs = ("The device works well but the update did not work ", "The update did not work but the device works well"),("The website was unresponsive but there were a wide selection of clothes",' There were a wide selection of clothes but the website was unresponsive'),('The website was good but customer service was unhelpful','Customer service was unhelpful but the website was good.')

In [47]:
# CHECK — Task 4 pairs
if my_pairs is None:
    print("[not yet] Task 4: replace `my_pairs = None` with at least 3 pairs.")
elif "lstm_model" not in globals():
    print("[not yet] Task 4 needs your Task 2 model trained first.")
else:
    assert len(my_pairs) >= 3, "write at least 3 pairs"
    for a, b in my_pairs:
        assert sorted(tokenize(a)) == sorted(tokenize(b)), f"not the same words:\n  {a}\n  {b}"
        assert a != b, "the two reviews in a pair must be in a different order"
        missing = unknown_words(a)
        if missing:
            print(f"   note: the LSTM has never seen {missing} — consider rewording")
    print(f"[ok] Task 4: {len(my_pairs)} valid pairs\n")
    compare([t for pair in my_pairs for t in pair])

   note: the LSTM has never seen ['device', 'works', 'well', 'work'] — consider rewording
   note: the LSTM has never seen ['website', 'unresponsive', 'there', 'wide', 'selection', 'clothes'] — consider rewording
   note: the LSTM has never seen ['website', 'good', 'unhelpful'] — consider rewording
[ok] Task 4: 3 valid pairs

probability of POSITIVE from each model
                                                                 review  single words  + word pairs  LSTM
                     The device works well but the update did not work          0.553         0.522 0.168
                      The update did not work but the device works well         0.553         0.522 0.261
The website was unresponsive but there were a wide selection of clothe…         0.612         0.590 0.986
 There were a wide selection of clothes but the website was unresponsi…         0.612         0.528 0.931
                The website was good but customer service was unhelpful         0.681         0.619 0.

### Your answer — Task 4

*The single words models were almost always split in the middle on every pair. This could be due to them not having any context and counting equal amounts of positive and negative sentiment. Surprisingly, LSTM got half of the sentiments wrong. For sentence 2, 4 and 6, the LSTM got the wrong sentiment. It seemed to repeat the setiment through the pair( negative for the first pair, positive for the other two pairs). 

A surprising result is the first pair for LSTM is negative while the other pair is positive. For me, the only noticable difference is that for the first pair, the negative sentiment( did not) is spelled out while in the other two pairs the negative was within the word('unhelpful','unresponsive'). Nevertheless, LSTM did not perform well. While there was a 0.1 difference in points within the pair, it was going in the wrong direction( the negative sentence had a higher Positive chance).*

SyntaxError: invalid syntax (4067511281.py, line 1)

---
## Task 5 — Swap the cell: LSTM → GRU

The GRU is the lighter design from the lecture: two gates instead of three.
Train a GRU version of your model — one word changes — and compare accuracy,
size, and training time. The embedding table and the head stay exactly as they
were; only the cell that carries the memory changes.

In [52]:
# TODO — replace None: set_seed(), then create your ReviewReader with cell=nn.GRU.

class ReviewReader(nn.Module):
    def __init__(self,vocab_size,cell=nn.GRU):
        super().__init__()
        self.embed=nn.Embedding(vocab_size, 64,padding_idx=0)#   1. turn word IDs into 64-number vectors      -> nn.Embedding(vocab_size, 64, padding_idx=0)
        self.reader = cell(64,64,batch_first=True)#   2. read them in order with a 64-number memory -> cell(64, 64, batch_first=True)
        self.head= nn.Linear(64,2)#   3. the head: turn the memory after the LAST word into 2 scores -> nn.Linear(64, 2)

    def forward(self,ids):
        word_vectors=self.embed(ids)
        outputs,final=self.reader(word_vectors)
        if isinstance(final, tuple):
            final= final[0]
        return self.head(final[-1])

set_seed()
gru_untrained = ReviewReader(len(vocab))

In [53]:
# CHECK + GIVEN — train and score the GRU.
if gru_untrained is None:
    print("[not yet] Task 5: replace `gru_untrained = None` with your GRU model.")
else:
    assert any(isinstance(m, nn.GRU) for m in gru_untrained.modules()), "use cell=nn.GRU"
    gru_model, gru_secs = train_reader(gru_untrained)
    slice_report("GRU reader", predict(gru_model, X_test), gru_secs, count(gru_model))
    print("[ok] Task 5\n")
    show_results()

[ok] Task 5

                            overall  simple  contrast  scope_short  scope_long  train seconds  learned numbers
model                                                                                                         
bag of words: single words   0.6453     1.0    0.5000       0.5327      0.5920           0.06              289
bag of words: + word pairs   0.7533     1.0    0.9291       0.5357      0.5987           0.09             1350
LSTM reader                  0.9992     1.0    1.0000       0.9970      1.0000           6.92            51970
GRU reader                   0.9983     1.0    1.0000       1.0000      0.9933          19.54            43650


### Your answer — Task 5

*For LSTM and GRU, both are almost identical on accuracy with LSTM being 0.0009 better, with almost identical short_scope of 0.003(GRU wins) and long_scope at 0.0067(LSTM wins). Most notably, the training time of LSTM(6.92s) beats GRU( 19.54s) by around 13 seconds. LSTM would be the best choice for shipping, due to almost identical accuracy and scope metrics, while being 300% faster in training. *

---
## Task 6 — The recommendation

### Your answer — Task 6: memo to the product owner

*Replace this text with your 300–400 word memo to Northwind's review-triage product owner, who does not read notebooks. It must contain all five of: (1) a clear recommendation — which model to run; (2) at least four numbers from this notebook, each with its slice or model; (3) the cost — training time and model size; (4) what share of real reviews would have to look like `simple` for your recommendation to change, and why; (5) one thing this evaluation cannot tell you, and what you would measure next. Write it yourself.*

To: Northwind Product Owner
From: Long Trinh
SUbject: Context Model Deployment

Dear Product Owner,
While bag of word models are helpful in identifying simple sentiments, they are not useful in processing sentences with mixed structure or a long scope. For longer queries, a LSTM model is the most fit for deployment. With our test set, this model performed at a 0.9992 accuracy overall. With a 100% score for simple and contrast sentence structures( 0.24 point increase compared to bag-of-word models), a 0.9997 for short scope(0.46 point increase ) and 1.000 for scope_long(0.4 increase). These results are on par with a GRU model while being 13 seconds faster(200% faster). 

This model takes around 7 seconds to train and runs at 51,970 learned numbers, 10,000 numbers more than the next best GRU model. 

This model is ideal for complex sentences where there are double meanings. However, in the case of a dataset dominated by simpler sentences, bag-of-word models like TfIDF bigrams could be more effective as they take around 0.09 seconds to train. The current dataset has around 5000 reviews with test reviews per slice: {'simple': 273, 'contrast': 296, 'scope_short': 336, 'scope_long': 299} ; with simple reviews at 0.2275 or around 23 %. At around 50% of the 'simple' slice, the accuracy for bag-of-word models might be sufficient for deployment. 

Something this model does not measure is the exact vectors that change the hidden state. For LSTMs and bidirectional LSTMs, the hiddenstate is a blackbox and it is difficult to trace the exact changes. 

While this model has certain unexplainabilties, it is highly efficient for long-sentence processing and is the best model so far for multi-sentiment sentence analysis. 